In [0]:
import delta

In [0]:
df_full = spark.read.format("parquet").load("/Volumes/raw/upsell/full_load/clientes/")

(df_full.coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.upsell.clientes"))

In [0]:
%sql
SELECT * 
FROM bronze.upsell.clientes

In [0]:
(spark.read
        .format("parquet")
        .load("/Volumes/raw/upsell/cdc/clientes/")
        .createOrReplaceTempView("clientes"))

In [0]:
query = '''
SELECT *
FROM clientes
QUALIFY ROW_NUMBER() OVER(PARTITION BY idCliente ORDER BY DtAtualizacao DESC) = 1
'''

df_cdc_unique = spark.sql(query)
df_cdc_unique.display()

In [0]:
bronze = delta.DeltaTable.forName(spark,"bronze.upsell.clientes")

# UPSERT
(bronze.alias("b")
        .merge(df_cdc_unique.alias("d"), "b.idCliente = d.idCliente")
        .whenMatchedDelete(condition= "d._operation = 'DELETE'")
        .whenMatchedUpdateAll(condition= "d._operation = 'UPDATE'")
        .whenNotMatchedInsertAll(condition="d._operation = 'INSERT' OR d._operation = 'UPDATE'")
        .execute()
)

In [0]:
%sql

SELECT * 
FROM bronze.upsell.clientes